# ENGE707 Facial Feature Classification — Task 1 & Task 2

Eight classes: `CatEye, DogEye, CatFace, DogFace, CatMouth, DogMouth, CatNose, DogNose`.

**Part A — Task 1: Dataset exploration**
1. Locate the data and inspect the folder structure
2. Link labels to image files
3. Class distribution
4. Sample images per class
5. One scan of every image: sizes, colour modes, corrupted files, duplicates, blur

**Part B — Task 2: Preprocessing and data pipeline**
6. Label encoding
7. Cleaning the training data
8. Train / validation split and class weights
9. Transforms, Dataset and DataLoaders
10. Sanity and leakage checks
11. Save outputs for the training notebook

All figures and tables are saved to `outputs/` (on Kaggle: `/kaggle/working/outputs/`).

**Run on Kaggle:** competition page → *Code* → *New Notebook* → *File → Import Notebook* → *Run All*.
No GPU needed. Full run ≈ 15–20 minutes (most of it is the image scan in section 5).

**Run locally:** set `DATA_DIR` to the folder containing `train_labels.csv` and the image folders,
and `OUT_DIR` to where outputs should go.

In [ ]:
import os
import io
import re
import json
import time
import random
import hashlib
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import sys
from sklearn.model_selection import StratifiedGroupKFold

os.environ["DATA_DIR"] = r"C:\Users\User\Documents\Data_Enginerring_AI\ENGE707-2026-S2-Facial-Feature-Classification-Challenge\enge-707-2026-s-2-cat-and-dog-facial-feature-classification-ch"
os.environ["OUT_DIR"] = "outputs"


# ---------------- Configuration ----------------
# Every setting that changes the data pipeline is set here and saved to outputs/preprocessing_config.json,
# so the training notebook can load exactly the same settings.
CFG = {
    "seed": 42,
    "img_size": 224,                 # model input height = width, in pixels
    "resize_mode": "pad",            # "pad": keep aspect ratio, pad to square | "stretch": resize straight to square
    "val_fraction": 0.15,            # approximate share of labelled images held out for validation
    "drop_corrupted": True,          # remove training files that cannot be decoded
    "drop_label_conflicts": True,    # remove duplicate groups whose copies have different labels
    "dedupe_exact_copies": True,     # keep only one copy of byte-identical training files with the same label
    "batch_size": 64,
    "num_workers": min(4, os.cpu_count() or 1) if sys.platform.startswith("linux") else 0,
    # ImageNet channel mean/std. These match ImageNet-pretrained models. If you train from scratch instead,
    # replace them with statistics computed on the TRAINING split only (never validation or test).
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
}

CLASSES = ["CatEye", "DogEye", "CatFace", "DogFace", "CatMouth", "DogMouth", "CatNose", "DogNose"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Where to look for the competition data. On Kaggle everything attached is mounted under /kaggle/input.
SEARCH_ROOT = Path(os.environ.get("DATA_DIR", "/kaggle/input"))
# Where figures and tables are written (Kaggle keeps anything in /kaggle/working).
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG["seed"])

# Plot style: one colour per series, light grid, no top/right spines.
BLUE, ORANGE, INK2 = "#2a78d6", "#eb6834", "#52514e"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "figure.dpi": 100,
})

def load_rgb(path):
    # Open an image for display; returns None instead of crashing if the file is corrupted.
    try:
        with Image.open(path) as im:
            return im.convert("RGB")
    except Exception:
        return None

# Part A — Task 1: Dataset exploration

## 1. Locate the data and inspect the folder structure

In [ ]:
def find_one(name):
    # Search recursively so this works whatever Kaggle names the mounted folder.
    hits = sorted(SEARCH_ROOT.rglob(name))
    if not hits:
        raise FileNotFoundError(
            f"{name} not found under {SEARCH_ROOT}. On Kaggle, attach the competition data with 'Add Input'."
        )
    return hits[0]

labels_path = find_one("train_labels.csv")
sample_sub_path = find_one("sample_submission.csv")
DATA_ROOT = labels_path.parent
print("Data root:", DATA_ROOT, "\n")

# Top-level contents: folders with their file counts, files with their sizes.
for p in sorted(DATA_ROOT.iterdir()):
    if p.is_dir():
        n = sum(1 for f in p.rglob("*") if f.is_file())
        print(f"[dir]  {p.name}/  ({n:,} files)")
    else:
        print(f"[file] {p.name}  ({p.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Index every image file under the data root.
all_imgs = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)
print(f"Image files found: {len(all_imgs):,}")
print("File extensions:", dict(Counter(p.suffix.lower() for p in all_imgs)))

def split_of(path):
    # Decide train/test from the folder names on the way to the file.
    folders = [s.lower() for s in path.relative_to(DATA_ROOT).parts[:-1]]
    if any("train" in s for s in folders):
        return "train"
    if any("test" in s for s in folders):
        return "test"
    return "unknown"

def id_from_name(path):
    # Filenames are expected to end in the numeric image_id, e.g. 123.jpg or img_00123.png.
    m = re.search(r"(\d+)$", path.stem)
    return int(m.group(1)) if m else None

img_df = pd.DataFrame({"path": all_imgs})
img_df["split"] = img_df["path"].map(split_of)
img_df["image_id"] = img_df["path"].map(id_from_name)

print("\nImages per split:\n", img_df["split"].value_counts().to_string())
for split, grp in img_df.groupby("split"):
    print(f"Example {split} files:", [str(p.relative_to(DATA_ROOT)) for p in grp["path"].head(3)])

# Stop early with a clear message if the filename pattern is not what we assumed.
assert (img_df["split"] != "unknown").all(), "Some images are not under a train/test folder - check the folder listing above."
assert img_df["image_id"].notna().all(), "Some filenames do not end in a number - edit id_from_name() to match the examples above."
assert not img_df.duplicated(["split", "image_id"]).any(), "Two files share the same id within a split - check the filename pattern."
img_df["image_id"] = img_df["image_id"].astype(int)

## 2. Load the label files and link labels to image files

In [ ]:
train_labels = pd.read_csv(labels_path)
sample_sub = pd.read_csv(sample_sub_path)

print("train_labels.csv:", train_labels.shape, "| columns:", list(train_labels.columns))
print("sample_submission.csv:", sample_sub.shape, "| columns:", list(sample_sub.columns))
display(train_labels.head())
display(sample_sub.head())

train_files = img_df.loc[img_df["split"] == "train", ["image_id", "path"]]
test_files = img_df.loc[img_df["split"] == "test", ["image_id", "path"]]

train_df = train_labels.merge(train_files, on="image_id", how="left")
test_df = sample_sub[["image_id"]].merge(test_files, on="image_id", how="left")

# Integrity checks: every label should have exactly one image and vice versa.
integrity = {
    "Train labels with no image file": int(train_df["path"].isna().sum()),
    "Train image files with no label": len(set(train_files["image_id"]) - set(train_labels["image_id"])),
    "Duplicate image_ids in train_labels.csv": int(train_labels["image_id"].duplicated().sum()),
    "Missing class values": int(train_labels["class"].isna().sum()),
    "Class names outside the 8 expected": sorted(set(train_labels["class"].dropna()) - set(CLASSES)),
    "Test ids with no image file": int(test_df["path"].isna().sum()),
    "Test image files not in sample_submission": len(set(test_files["image_id"]) - set(sample_sub["image_id"])),
}
for k, v in integrity.items():
    print(f"{k:45s} {v}")

# Keep only rows that actually have an image for the rest of the analysis.
train_df = train_df.dropna(subset=["path"]).reset_index(drop=True)
test_df_all = test_df.copy()             # kept to check the submission covers every test id (Part B)
test_df = test_df.dropna(subset=["path"]).reset_index(drop=True)
print(f"\nUsable train images: {len(train_df):,}   Usable test images: {len(test_df):,}")

## 3. Class distribution

In [ ]:
class_counts = train_df["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
dist = pd.DataFrame({"count": class_counts, "percent": (100 * class_counts / class_counts.sum()).round(2)})
display(dist)
print(f"Largest / smallest class ratio: {class_counts.max() / max(class_counts.min(), 1):.2f}")

# The 8 labels are really two factors: animal (Cat/Dog) x facial feature (Eye/Face/Mouth/Nose).
train_df["animal"] = train_df["class"].str.extract(r"^(Cat|Dog)", expand=False)
train_df["feature"] = train_df["class"].str.replace(r"^(Cat|Dog)", "", regex=True)
by_factor = pd.crosstab(train_df["feature"], train_df["animal"], margins=True, margins_name="Total")
display(by_factor)

dist.to_csv(OUT_DIR / "class_counts.csv")
by_factor.to_csv(OUT_DIR / "class_counts_animal_x_feature.csv")

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(dist.index, dist["count"], color=BLUE, width=0.6)
ax.bar_label(bars, labels=[f"{c:,}" for c in dist["count"]], padding=3, fontsize=9, color=INK2)
ax.set_ylabel("Training images")
ax.set_title("Training images per class", loc="left")
ax.grid(axis="x", visible=False)
ax.margins(y=0.12)
fig.tight_layout()
fig.savefig(OUT_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Your observations (class balance):** _write what you notice here in your own words_

## 4. Sample images from each class

In [ ]:
N_PER_CLASS = 6
fig, axes = plt.subplots(len(CLASSES), N_PER_CLASS, figsize=(N_PER_CLASS * 1.9, len(CLASSES) * 2.0))
for r, cls in enumerate(CLASSES):
    pool = train_df.loc[train_df["class"] == cls, ["image_id", "path"]]
    picks = pool.sample(min(N_PER_CLASS, len(pool)), random_state=CFG["seed"])
    for c in range(N_PER_CLASS):
        ax = axes[r, c]
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
        if c < len(picks):
            row = picks.iloc[c]
            im = load_rgb(row["path"])
            if im is None:
                ax.text(0.5, 0.5, "unreadable", ha="center", va="center", fontsize=8, color=INK2)
                ax.set_title(f"id {row['image_id']}", fontsize=7, color=INK2)
            else:
                ax.imshow(im)
                ax.set_title(f"id {row['image_id']}  {im.size[0]}x{im.size[1]}", fontsize=7, color=INK2)
    axes[r, 0].set_ylabel(cls, fontsize=10, rotation=0, ha="right", va="center", labelpad=10)
fig.suptitle("Random training samples per class (title = image id and width x height)", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "samples_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

**Your observations (what the classes look like, anything confusing):** _write here_

## 5. Scan every image (train and test)

Each file is read **once** and we record:
- `width`, `height`, `mode` (RGB / L = greyscale / RGBA ...), file size
- `ok` — whether the file can be verified and fully decoded (False = corrupted)
- `md5` — hash of the raw bytes → finds byte-identical duplicates
- `dhash` — difference hash of a 9×8 greyscale thumbnail → the same picture saved at a different size or
  compression gets the same hash, so it also finds near-duplicates (used for the split in Part B)
- `blur` — variance of the Laplacian of the greyscale image. Low = few sharp edges = likely blurry.
  It also depends on image size and content, so it is used to *rank* images for a visual check, not as a hard rule.

In [ ]:
def dhash_hex(img, size=8):
    g = np.asarray(img.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()        # is each pixel brighter than its left neighbour?
    return np.packbits(bits).tobytes().hex()

def inspect_image(path):
    rec = {"path": path, "file_kb": np.nan, "md5": None, "dhash": None, "ok": False,
           "width": np.nan, "height": np.nan, "mode": None, "blur": np.nan, "error": None}
    try:
        data = Path(path).read_bytes()
        rec["file_kb"] = len(data) / 1024
        rec["md5"] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            im.verify()                      # structural integrity check
        with Image.open(io.BytesIO(data)) as im:
            im.load()                        # full decode (catches truncated files)
            rec["width"], rec["height"] = im.size
            rec["mode"] = im.mode
            rec["dhash"] = dhash_hex(im)
            gray = np.asarray(im.convert("L"), dtype=np.uint8)
        rec["blur"] = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        rec["ok"] = True
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"
    return rec

scan_input = pd.concat([
    train_df[["image_id", "path", "class"]].assign(split="train"),
    test_df[["image_id", "path"]].assign(split="test", **{"class": None}),
], ignore_index=True)

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    records = list(pool.map(inspect_image, scan_input["path"], chunksize=256))
stats = scan_input.merge(pd.DataFrame(records), on="path")
stats["aspect"] = stats["width"] / stats["height"]
stats.drop(columns=["path"]).to_csv(OUT_DIR / "image_stats.csv", index=False)
print(f"Scanned {len(stats):,} images in {time.time() - t0:.0f}s. Per-image table: {OUT_DIR / 'image_stats.csv'}")

# Attach the scan results to the training table (used in Part B).
scan_cols = ["path", "ok", "md5", "dhash", "width", "height", "mode", "blur", "error"]
train_df = train_df.merge(stats.loc[stats["split"] == "train", scan_cols], on="path", how="left")

### 5a. Image sizes and colour modes

In [ ]:
ok_stats = stats[stats["ok"]]
print("Colour modes by split:")
display(pd.crosstab(ok_stats["mode"], ok_stats["split"]))

print("Width / height / aspect / file size summary by split:")
display(ok_stats.groupby("split")[["width", "height", "aspect", "file_kb"]].describe().T.round(2))

n_sizes = {s: g[["width", "height"]].drop_duplicates().shape[0] for s, g in ok_stats.groupby("split")}
print("Number of distinct (width, height) combinations:", n_sizes)
top_sizes = (ok_stats.groupby(["width", "height"]).size().sort_values(ascending=False)
             .head(10).rename("images").reset_index())
print("Most common sizes (train + test):")
display(top_sizes)

print("Median width x height per class (train):")
size_by_class = ok_stats[ok_stats["split"] == "train"].groupby("class")[["width", "height"]].median().reindex(CLASSES)
display(size_by_class)
size_by_class.to_csv(OUT_DIR / "median_size_per_class.csv")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, col in zip(axes[:2], ["width", "height"]):
    hi = ok_stats[col].quantile(0.995)  # clip extreme outliers so the shape is readable
    bins = np.linspace(ok_stats[col].min(), hi, 50)
    for split, color in [("train", BLUE), ("test", ORANGE)]:
        ax.hist(ok_stats.loc[ok_stats["split"] == split, col].clip(upper=hi), bins=bins, color=color,
                histtype="step", linewidth=1.8, label=split, density=True)
    ax.set_title(f"Image {col} (px)", loc="left")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
sample_pts = ok_stats.sample(min(len(ok_stats), 8000), random_state=CFG["seed"])
for split, color in [("train", BLUE), ("test", ORANGE)]:
    s = sample_pts[sample_pts["split"] == split]
    axes[2].scatter(s["width"], s["height"], s=8, alpha=0.35, color=color, label=split, linewidths=0)
axes[2].set_xlabel("Width (px)"); axes[2].set_ylabel("Height (px)")
axes[2].set_title("Width vs height (sample of 8,000)", loc="left")
axes[2].legend(frameon=False, markerscale=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "image_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

**Your observations (sizes, modes, what input size might suit the model):** _write here_

### 5b. Corrupted / unreadable files

In [ ]:
bad = stats[~stats["ok"]]
print(f"Unreadable files: {len(bad)} (train {int((bad['split'] == 'train').sum())}, test {int((bad['split'] == 'test').sum())})")
if len(bad):
    display(bad[["split", "image_id", "class", "error"]].head(20))
    bad[["split", "image_id", "class", "error"]].to_csv(OUT_DIR / "corrupted_files.csv", index=False)

### 5c. Exact duplicate files

In [ ]:
good = stats[stats["ok"]].copy()
dup_groups = good.groupby("md5").filter(lambda g: len(g) > 1)

train_dup = good[good["split"] == "train"]
train_dup = train_dup[train_dup.duplicated("md5", keep=False)]
test_dup = good[good["split"] == "test"]
test_dup = test_dup[test_dup.duplicated("md5", keep=False)]

# Duplicates inside train whose copies carry different labels (label noise).
conflicts_md5 = train_dup.groupby("md5").filter(lambda g: g["class"].nunique() > 1)

# Files that appear in both train and test. Counted ONLY as a data-quality / leakage check for the report.
# Never copy train labels onto these test ids - inferring hidden test labels breaks the competition rules.
cross = set(good.loc[good["split"] == "train", "md5"]) & set(good.loc[good["split"] == "test", "md5"])

dup_summary = {
    "Train images that have an exact duplicate in train": len(train_dup),
    "  ...distinct duplicated train files": train_dup["md5"].nunique(),
    "  ...duplicate groups with conflicting labels": conflicts_md5["md5"].nunique(),
    "Test images that have an exact duplicate in test": len(test_dup),
    "Distinct files present in both train and test": len(cross),
}
for k, v in dup_summary.items():
    print(f"{k:55s} {v}")

dup_groups[["split", "image_id", "class", "md5"]].sort_values(["md5", "split"]).to_csv(
    OUT_DIR / "exact_duplicates.csv", index=False)

# Show up to 4 duplicate groups from train (conflicting-label groups first).
show = list(conflicts_md5["md5"].unique()[:4])
show += [m for m in train_dup["md5"].unique() if m not in show][: 4 - len(show)]
if show:
    fig, axes = plt.subplots(len(show), 4, figsize=(8, 2.2 * len(show)), squeeze=False)
    for r, m in enumerate(show):
        grp = train_dup[train_dup["md5"] == m].head(4)
        for c in range(4):
            ax = axes[r, c]; ax.axis("off")
            if c < len(grp):
                row = grp.iloc[c]
                ax.imshow(load_rgb(row["path"]))
                ax.set_title(f"id {row['image_id']}\n{row['class']}", fontsize=8, color=INK2)
    fig.suptitle("Exact duplicate groups in train (one row per group)", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "duplicate_examples.png", dpi=150, bbox_inches="tight")
    plt.show()

**Your observations (duplicates, label conflicts):** _write here_

### 5d. Blur check

In [ ]:
train_ok = stats[(stats["split"] == "train") & stats["ok"]].copy()

print("Blur score (variance of Laplacian) by class - lower = blurrier:")
blur_by_class = train_ok.groupby("class")["blur"].describe(percentiles=[0.05, 0.5]).reindex(CLASSES).round(1)
display(blur_by_class)
blur_by_class.to_csv(OUT_DIR / "blur_by_class.csv")

fig, ax = plt.subplots(figsize=(9, 3.8))
data = [np.log10(train_ok.loc[train_ok["class"] == c, "blur"].clip(lower=1e-3)) for c in CLASSES]
ax.boxplot(data, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#cde2fb", edgecolor=BLUE), medianprops=dict(color=BLUE, linewidth=2),
           whiskerprops=dict(color=INK2), capprops=dict(color=INK2))
ax.set_xticks(range(1, len(CLASSES) + 1))
ax.set_xticklabels(CLASSES)  # set separately: boxplot's labels argument was renamed across matplotlib versions
ax.set_ylabel("log10(blur score)")
ax.set_title("Sharpness by class (higher = sharper)", loc="left")
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "blur_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

# The 16 lowest-scoring training images, for a visual check.
low = train_ok.nsmallest(16, "blur")
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, (_, row) in zip(axes.ravel(), low.iterrows()):
    ax.imshow(load_rgb(row["path"]))
    ax.set_title(f"{row['class']}\nid {row['image_id']}  blur {row['blur']:.1f}", fontsize=7, color=INK2)
    ax.axis("off")
for ax in axes.ravel()[len(low):]:
    ax.axis("off")
fig.suptitle("16 lowest blur scores in train", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "blurriest_train_images.png", dpi=150, bbox_inches="tight")
plt.show()

**Your observations (are low-score images actually blurry? any classes affected more?):** _write here_

**How the data will be used (write your own):** training / validation / Kaggle submission, and anything above
that affects that plan.

# Part B — Task 2: Preprocessing and data pipeline

## 6. Encode class labels

The class names are mapped to integers with a **fixed, explicit order** (`CLASSES` in the setup cell), not
whatever order the data happens to appear in. The same mapping is saved to `class_mapping.json` and is used
later to turn predicted integers back into class names for the Kaggle submission.

In [ ]:
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

unknown = sorted(set(train_df["class"]) - set(CLASSES))
assert not unknown, f"Unexpected class names in train_labels.csv: {unknown}"
train_df["label"] = train_df["class"].map(CLASS_TO_IDX).astype(int)

display(pd.DataFrame({"class": CLASSES, "label": [CLASS_TO_IDX[c] for c in CLASSES]}).set_index("label"))

with open(OUT_DIR / "class_mapping.json", "w") as f:
    json.dump({"class_to_idx": CLASS_TO_IDX, "idx_to_class": {str(k): v for k, v in IDX_TO_CLASS.items()}}, f, indent=2)

## 7. Clean the training data

Using the scan from section 5:
- **corrupted** files are removed (they cannot be loaded);
- **duplicate groups with conflicting labels** are removed (we cannot know which label is correct);
- **extra byte-identical copies** with the same label are removed (one copy is kept).

Each image is also given a **group** = its dHash (near-duplicates share a group). Images with a "flat" hash
(no brightness changes at all) use their md5 instead, so unrelated plain images are not grouped together.
Every removed image is logged with its reason in `removed_training_images.csv`.

In [ ]:
FLAT_HASH = "0" * 16
train_df["group"] = np.where(
    train_df["dhash"].isna() | (train_df["dhash"] == FLAT_HASH),
    "md5_" + train_df["md5"].fillna(train_df["image_id"].astype(str)),
    "dh_" + train_df["dhash"].fillna(""),
)

removed = []   # every image dropped from training, with the reason

# 7a. Corrupted files
corrupt_train = train_df[~train_df["ok"].fillna(False).astype(bool)]
print(f"Corrupted / unreadable training files: {len(corrupt_train)}")
if CFG["drop_corrupted"] and len(corrupt_train):
    removed.append(corrupt_train.assign(reason="corrupted"))
    train_df = train_df.drop(index=corrupt_train.index)

# 7b. Duplicate groups (exact or near) whose members carry different labels
labels_per_group = train_df.groupby("group")["class"].nunique()
conflict_groups = labels_per_group[labels_per_group > 1].index
conflicts = train_df[train_df["group"].isin(conflict_groups)]
print(f"Duplicate groups with conflicting labels: {len(conflict_groups)}  ({len(conflicts)} images)")
if len(conflicts):
    display(conflicts.sort_values("group")[["image_id", "class", "group"]].head(20))
if CFG["drop_label_conflicts"] and len(conflicts):
    removed.append(conflicts.assign(reason="duplicate with conflicting label"))
    train_df = train_df[~train_df["group"].isin(conflict_groups)]

# 7c. Byte-identical copies with the same label: keep the first one
copies = train_df[train_df.duplicated(["md5", "class"], keep="first")]
print(f"Extra byte-identical copies (same label): {len(copies)}")
if CFG["dedupe_exact_copies"] and len(copies):
    removed.append(copies.assign(reason="exact duplicate copy"))
    train_df = train_df.drop(index=copies.index)

removed_df = (pd.concat(removed, ignore_index=True) if removed
              else pd.DataFrame(columns=["image_id", "class", "reason", "error"]))
removed_df[["image_id", "class", "reason", "error"]].to_csv(OUT_DIR / "removed_training_images.csv", index=False)

train_df = train_df.reset_index(drop=True)
group_sizes = train_df["group"].value_counts()
print(f"\nImages kept for training/validation: {len(train_df):,}  (removed {len(removed_df):,})")
print(f"Groups: {len(group_sizes):,}   groups with >1 image: {(group_sizes > 1).sum():,}   largest group: {group_sizes.max()}")
if len(removed_df):
    display(removed_df["reason"].value_counts().rename("images removed"))

In [ ]:
# Visual check of near-duplicate groups (same dHash, different files). If these do NOT look like the same
# picture, the hash is grouping too aggressively.
near = train_df[train_df["group"].str.startswith("dh_")]
near = near[near.groupby("group")["md5"].transform("nunique") > 1]
print(f"Near-duplicate groups (different bytes, same dHash): {near['group'].nunique()}")
show_groups = (near["group"].drop_duplicates().sample(min(4, near["group"].nunique()), random_state=CFG["seed"]).tolist()
               if len(near) else [])

if show_groups:
    fig, axes = plt.subplots(len(show_groups), 4, figsize=(8, 2.2 * len(show_groups)), squeeze=False)
    for r, g in enumerate(show_groups):
        grp = near[near["group"] == g].head(4)
        for c in range(4):
            ax = axes[r, c]; ax.axis("off")
            if c < len(grp):
                row = grp.iloc[c]
                ax.imshow(load_rgb(row["path"]))
                ax.set_title(f"id {row['image_id']} {row['class']}\n{int(row['width'])}x{int(row['height'])}",
                             fontsize=7, color=INK2)
    fig.suptitle("Near-duplicate groups (one row per group)", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "near_duplicate_groups.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Train / validation split

`StratifiedGroupKFold` splits the cleaned data into `round(1 / val_fraction)` folds:
- **stratified** → every fold has (almost) the same class proportions as the full dataset;
- **grouped** → all images in one hash group go to the same fold, so the same or a near-identical image
  can never be in training and validation at the same time.

Fold 0 is the validation set. Every image's fold number is saved, so the other folds can be used later for
cross-validation without re-running this notebook. The test images are never part of this split.

What this grouping does **not** catch: different crops that come from the same original photo
(for example an eye crop and the face crop of the same animal) have different hashes, so they can still be
split across train and validation. The data has no id that links crops to their source photo.

**Your justification (write your own):** _why this split strategy and validation size_

In [ ]:
n_splits = max(2, round(1 / CFG["val_fraction"]))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CFG["seed"])
train_df["fold"] = -1
t0 = time.time()
for k, (_, idx) in enumerate(sgkf.split(train_df, train_df["label"], groups=train_df["group"])):
    train_df.loc[train_df.index[idx], "fold"] = k
print(f"Assigned {n_splits} folds in {time.time() - t0:.0f}s")
assert (train_df["fold"] >= 0).all()

VAL_FOLD = 0
train_split = train_df[train_df["fold"] != VAL_FOLD].reset_index(drop=True)
val_split = train_df[train_df["fold"] == VAL_FOLD].reset_index(drop=True)

# Leakage checks: no image, no identical file and no duplicate group on both sides.
assert set(train_split["image_id"]).isdisjoint(val_split["image_id"]), "image in both train and val"
assert set(train_split["group"]).isdisjoint(val_split["group"]), "duplicate group in both train and val"
assert set(train_split["md5"]).isdisjoint(val_split["md5"]), "identical file in both train and val"
print(f"Train: {len(train_split):,}   Validation: {len(val_split):,}   ({100 * len(val_split) / len(train_df):.1f}% validation)")
print("Leakage checks passed: no shared image ids, md5 hashes or dHash groups between train and validation.")

split_table = pd.DataFrame({
    "train": train_split["class"].value_counts(),
    "val": val_split["class"].value_counts(),
}).reindex(CLASSES)
split_table["train %"] = (100 * split_table["train"] / split_table["train"].sum()).round(2)
split_table["val %"] = (100 * split_table["val"] / split_table["val"].sum()).round(2)
display(split_table)
split_table.to_csv(OUT_DIR / "split_class_distribution.csv")

xpos = np.arange(len(CLASSES)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(xpos - w / 2, split_table["train %"], w * 0.95, color=BLUE, label="train")
ax.bar(xpos + w / 2, split_table["val %"], w * 0.95, color=ORANGE, label="validation")
ax.set_xticks(xpos); ax.set_xticklabels(CLASSES)
ax.set_ylabel("% of split")
ax.set_title("Class proportions in the training and validation splits", loc="left")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "split_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Class weights (computed on the training split only)

Weights `N / (num_classes × count_c)` for an optional weighted loss in the training notebook. They use the
**training split only**, so nothing about the validation set leaks into training.

In [ ]:
train_counts = train_split["label"].value_counts().sort_index()
class_weights = (len(train_split) / (len(CLASSES) * train_counts)).round(4)
display(pd.DataFrame({"class": CLASSES, "train count": train_counts.values, "weight": class_weights.values}))
with open(OUT_DIR / "class_weights.json", "w") as f:
    json.dump({IDX_TO_CLASS[int(i)]: float(wt) for i, wt in class_weights.items()}, f, indent=2)

## 9. Transforms, Dataset and DataLoaders

**Resize** — images come in many sizes and shapes. With `resize_mode="pad"` each image is scaled so its
longer side equals `img_size` (aspect ratio kept), then padded to a square with the ImageNet mean colour
(which becomes ~0 after normalisation). With `"stretch"` it is resized straight to a square.

**Normalise** — `ToTensor()` rescales pixels from 0–255 to 0–1, then `Normalize(mean, std)` standardises each
channel.

**Augment (training only)** — random horizontal flip, small rotation / shift / zoom, and mild colour jitter.
No vertical flips (faces are upright in real photos).

**Consistency** — validation and test images use the *same* `eval_tf` (resize + normalise only, no randomness),
built by the same function as the training transform.

**Your justification (write your own):** _input size, pad vs stretch, which augmentations and why_

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

MEAN_RGB = tuple(int(round(m * 255)) for m in CFG["mean"])   # padding colour = ImageNet mean

class ResizePad:
    # Resize so the longer side equals `size` (keeps aspect ratio), then pad to a size x size square.
    def __init__(self, size, fill=MEAN_RGB):
        self.size, self.fill = size, fill

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
        img = img.resize((nw, nh), Image.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(img, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

    def __repr__(self):
        return f"ResizePad(size={self.size})"

def build_transforms(cfg, train):
    size = cfg["img_size"]
    if cfg["resize_mode"] == "pad":
        resize = ResizePad(size)
    else:
        resize = T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC)
    steps = [resize]
    if train:
        steps += [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1), fill=MEAN_RGB),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        ]
    steps += [T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])]
    return T.Compose(steps)

train_tf = build_transforms(CFG, train=True)
eval_tf = build_transforms(CFG, train=False)    # used for BOTH validation and test
print("Train transform:\n", train_tf, "\n\nValidation / test transform:\n", eval_tf)

In [ ]:
class FacialFeatureDataset(Dataset):
    # Returns (image_tensor, label) for labelled data, or (image_tensor, image_id) for the test set.
    def __init__(self, df, transform, labelled=True, img_size=CFG["img_size"]):
        self.paths = df["path"].astype(str).tolist()
        self.ids = df["image_id"].astype(int).tolist()
        self.labels = df["label"].astype(int).tolist() if labelled else None
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as im:
                img = im.convert("RGB")             # handles greyscale, RGBA and palette images
        except Exception:
            # Only possible for test images (corrupted training files were removed in section 7).
            # A blank image still gets a prediction, so the submission keeps every test id.
            img = Image.new("RGB", (self.img_size, self.img_size), MEAN_RGB)
        x = self.transform(img)
        target = self.ids[i] if self.labels is None else self.labels[i]
        return x, target

def seed_worker(worker_id):
    # Gives every DataLoader worker its own reproducible random seed (for the random augmentations).
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)

def make_loader(ds, shuffle, batch_size, seed=CFG["seed"]):
    g = torch.Generator()
    g.manual_seed(seed)
    nw = CFG["num_workers"]
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle,
        num_workers=nw, pin_memory=torch.cuda.is_available(),
        persistent_workers=nw > 0, worker_init_fn=seed_worker, generator=g,
    )

assert len(test_df) == len(test_df_all), "Some test ids have no image file - the submission would be incomplete."

torch.manual_seed(CFG["seed"])
train_ds = FacialFeatureDataset(train_split, train_tf)
val_ds = FacialFeatureDataset(val_split, eval_tf)
test_ds = FacialFeatureDataset(test_df, eval_tf, labelled=False)

train_loader = make_loader(train_ds, shuffle=True, batch_size=CFG["batch_size"])
val_loader = make_loader(val_ds, shuffle=False, batch_size=CFG["batch_size"] * 2)
test_loader = make_loader(test_ds, shuffle=False, batch_size=CFG["batch_size"] * 2)

print(f"train: {len(train_ds):,} images, {len(train_loader)} batches")
print(f"val:   {len(val_ds):,} images, {len(val_loader)} batches")
print(f"test:  {len(test_ds):,} images, {len(test_loader)} batches")

## 10. Sanity and leakage checks

In [ ]:
def denorm(x):
    # Undo Normalize so a tensor can be shown as an image.
    mean = torch.tensor(CFG["mean"]).view(3, 1, 1)
    std = torch.tensor(CFG["std"]).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# One batch from each loader: shape, dtype, value range and labels.
for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    xb, yb = next(iter(loader))
    print(f"{name:5s} x {tuple(xb.shape)} {xb.dtype}  min {xb.min():.2f}  max {xb.max():.2f}  "
          f"mean {xb.mean():.2f}  std {xb.std():.2f} | target {tuple(yb.shape)} {yb.dtype}")
    assert xb.shape[1:] == (3, CFG["img_size"], CFG["img_size"])
    if name != "test":
        assert int(yb.min()) >= 0 and int(yb.max()) < len(CLASSES)

In [ ]:
# Augmentation preview: one validation image per class. Column 1 = original file, column 2 = what validation/test
# images look like after preprocessing, columns 3+ = random training augmentations of the same image.
N_AUG = 5
rows = val_split.groupby("class").head(1).set_index("class").reindex(CLASSES).dropna(subset=["path"])
fig, axes = plt.subplots(len(rows), N_AUG + 2, figsize=(1.9 * (N_AUG + 2), 2.1 * len(rows)), squeeze=False)
for r, (cls, row) in enumerate(rows.iterrows()):
    img = load_rgb(row["path"])
    panels = [("original", np.asarray(img)), ("val/test", denorm(eval_tf(img)))]
    panels += [(f"aug {k + 1}", denorm(train_tf(img))) for k in range(N_AUG)]
    for c, (title, arr) in enumerate(panels):
        axes[r, c].imshow(arr); axes[r, c].axis("off")
        axes[r, c].set_title(title if r == 0 else "", fontsize=8, color=INK2)
    axes[r, 0].text(-0.08, 0.5, cls, transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "augmentation_preview.png", dpi=150, bbox_inches="tight")
plt.show()

# A real training batch exactly as the model will receive it.
xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, x, y in zip(axes.ravel(), xb[:16], yb[:16]):
    ax.imshow(denorm(x)); ax.axis("off")
    ax.set_title(IDX_TO_CLASS[int(y)], fontsize=8, color=INK2)
fig.suptitle("First 16 images of a training batch (after augmentation)", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "train_batch_preview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Loading speed: if this is slow, the GPU will sit idle during training waiting for data.
n_batches = min(30, len(train_loader))
t0 = time.time()
for i, (xb, yb) in enumerate(train_loader):
    if i + 1 >= n_batches:
        break
elapsed = time.time() - t0
print(f"{n_batches} training batches in {elapsed:.1f}s  ->  {n_batches * CFG['batch_size'] / elapsed:,.0f} images/s "
      f"with {CFG['num_workers']} workers")

In [ ]:
# Final leakage / consistency checklist.
leak_checks = {
    "No image id in both train and validation": set(train_split["image_id"]).isdisjoint(val_split["image_id"]),
    "No duplicate group in both train and validation": set(train_split["group"]).isdisjoint(val_split["group"]),
    "Test images are not used for training or validation": len(set(test_df["path"].astype(str))
                                                               & set(train_df["path"].astype(str))) == 0,
    "Validation and test share the same transform object": val_ds.transform is test_ds.transform,
    "Validation/test transform has no random steps": not any("Random" in type(t).__name__ or "Jitter" in type(t).__name__
                                                             for t in eval_tf.transforms),
    "Normalisation uses fixed constants (not fitted on any split)": True,
    "Class weights computed from the training split only": int(train_counts.sum()) == len(train_split),
    "Every test id has an image": len(test_ds) == len(sample_sub),
}
for k, v in leak_checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
assert all(leak_checks.values())

## 11. Save everything the training notebook needs

In [ ]:
def rel(p):
    return str(Path(p).relative_to(DATA_ROOT))

cols = ["image_id", "class", "label", "fold", "group", "md5"]
splits_out = train_df[cols].assign(relpath=train_df["path"].map(rel), is_val=train_df["fold"] == VAL_FOLD)
splits_out.to_csv(OUT_DIR / "splits.csv", index=False)

with open(OUT_DIR / "preprocessing_config.json", "w") as f:
    json.dump({**CFG, "classes": CLASSES, "val_fold": VAL_FOLD, "n_splits": n_splits,
               "n_train": len(train_split), "n_val": len(val_split), "n_test": len(test_df),
               "train_transform": str(train_tf), "eval_transform": str(eval_tf)}, f, indent=2)

for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:36s} {f.stat().st_size / 1024:8.1f} KB")